# OpenVoice V2 Voice Conversion
Converts TeraTTS (Natasha) audio to a target voice using OpenVoice V2 tone color transfer.

**Input:** WAV files from TeraTTS + reference voice sample
**Output:** Voice-converted WAVs + combined MP3, saved to Google Drive

In [ ]:
#@title 1. Mount Drive + Install OpenVoice V2
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
OUTPUT_DIR = '/content/drive/My Drive/tts_voice_converted'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Drive mounted!')

# Install OpenVoice
!pip install -q git+https://github.com/myshell-ai/OpenVoice.git

# Download checkpoints
!git clone https://github.com/myshell-ai/OpenVoice.git openvoice_repo
!wget -q https://myshell-public-repo-host.s3.amazonaws.com/openvoice/checkpoints_v2_0417.zip
!unzip -q -o checkpoints_v2_0417.zip

print('OpenVoice V2 installed!')

In [ ]:
#@title 2. Upload source WAVs + reference voice
# Clone data repo for the TeraTTS WAVs
!git clone https://github.com/stuk88/post-scarcity-architecture.git data_repo 2>/dev/null || echo 'Already cloned'

# Upload reference voice (the voice you want the output to sound like)
from google.colab import files
print('Upload your reference voice WAV (the voice you want):')
uploaded = files.upload()
ref_name = list(uploaded.keys())[0]
import shutil
shutil.move(ref_name, 'reference_voice.wav')
print(f'Reference voice saved: reference_voice.wav')

In [ ]:
#@title 3. Upload TeraTTS WAV files
# Upload the WAV files generated by TeraTTS locally
from google.colab import files
import os

os.makedirs('source_wavs', exist_ok=True)
print('Upload all TeraTTS WAV files from tts_raw/ (select multiple):')
uploaded = files.upload()
for name, content in uploaded.items():
    with open(f'source_wavs/{name}', 'wb') as f:
        f.write(content)
    print(f'  {name} ({len(content)/(1024*1024):.1f} MB)')
print(f'\n{len(uploaded)} files uploaded')

In [ ]:
#@title 4. Convert all chapters to target voice
import os
import gc
import time
import torch
from pathlib import Path
from openvoice import se_extractor
from openvoice.api import ToneColorConverter

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
OUTPUT = Path('/content/drive/My Drive/tts_voice_converted')

# Load converter
print('Loading ToneColorConverter...')
converter = ToneColorConverter('checkpoints_v2/converter/config.json', device=device)
converter.load_ckpt('checkpoints_v2/converter/checkpoint.pth')
print('Converter loaded!')

# Extract target voice embedding (the voice you want)
print('Extracting target voice embedding...')
target_se, _ = se_extractor.get_se('reference_voice.wav', converter, vad=True)
print('Target SE extracted!')

# Extract source voice embedding (Natasha / TeraTTS)
# Use the first WAV as source SE reference
source_wavs = sorted(Path('source_wavs').glob('*.wav'))
print(f'\nExtracting source voice embedding from {source_wavs[0].name}...')
source_se, _ = se_extractor.get_se(str(source_wavs[0]), converter, vad=True)
print('Source SE extracted!')

# Convert each chapter
print(f'\nConverting {len(source_wavs)} files...\n')
t0 = time.time()

for i, src_wav in enumerate(source_wavs):
    out_wav = OUTPUT / src_wav.name
    if out_wav.exists():
        print(f'  [skip] {src_wav.name}')
        continue

    print(f'  [{i+1}/{len(source_wavs)}] {src_wav.name}', end=' ')

    try:
        converter.convert(
            audio_src_path=str(src_wav),
            src_se=source_se,
            tgt_se=target_se,
            output_path=str(out_wav),
            message='@MyShell',
        )
        size = out_wav.stat().st_size / (1024*1024)
        elapsed = time.time() - t0
        print(f'-> {size:.1f} MB [{elapsed:.0f}s]')
    except Exception as e:
        print(f'ERROR: {str(e)[:100]}')

    gc.collect()
    torch.cuda.empty_cache()

print(f'\nDone! {time.time()-t0:.0f}s total')
print(f'Files saved to Google Drive: tts_voice_converted/')

In [ ]:
#@title 5. Combine into final MP3
import subprocess
from pathlib import Path

OUTPUT = Path('/content/drive/My Drive/tts_voice_converted')
wav_files = sorted(OUTPUT.glob('*.wav'))
print(f'Combining {len(wav_files)} chapters...')

with open('/tmp/filelist.txt', 'w') as f:
    for wav in wav_files:
        f.write(f"file '{wav}'\n")

mp3_path = OUTPUT / 'Post_Scarcity_v3_ru_final.mp3'
result = subprocess.run([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
    '-i', '/tmp/filelist.txt',
    '-codec:a', 'libmp3lame', '-qscale:a', '2',
    str(mp3_path)
], capture_output=True, text=True)

if result.returncode == 0:
    size = mp3_path.stat().st_size / (1024*1024)
    print(f'Saved: {mp3_path.name} ({size:.1f} MB)')
    print('File is in Google Drive: tts_voice_converted/')
else:
    print(f'Error: {result.stderr[:300]}')